In [1]:
import pymysql
import csv


In [2]:

con = pymysql.connect(
    host='localhost', 
    user='root', 
    password='root', 
    port=3306, 
    db='pokemon151_move_db', 
    charset='utf8'
)
cursor = con.cursor()


In [3]:

# 导入CSV数据到MySQL表
def import_csv_to_table(cursor, csv_file, table_name):
    with open(csv_file, 'r', newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)

        columns = ', '.join([f"`{col}`" for col in header])
        placeholders = ', '.join(['%s'] * len(header))
        sql = f"INSERT INTO `{table_name}` ({columns}) VALUES ({placeholders})"

        batch = []
        for row in reader:
            # 统一缺省值为None
            row = [None if val == '' or val == '—' else val for val in row]
            batch.append(row)
            if len(batch) >= 1000:
                cursor.executemany(sql, batch)
                cursor.connection.commit()
                batch = []

        if batch:
            cursor.executemany(sql, batch)
            cursor.connection.commit()


tables = [
    ('pokemon', r'../data/processed/pokemon.csv'),
    ('move',    r'../data/processed/move.csv'),
    ('learn',   r'../data/processed/learn.csv'),
]

for table_name, csv_file in tables:
    print('导入表：', table_name)
    import_csv_to_table(cursor, csv_file, table_name)

cursor.close()
con.close()
print("全部导入完成")

导入表： pokemon
导入表： move
导入表： learn
全部导入完成
